In [1]:
!git clone -q https://github.com/nalinwadhwa02/CCS521FA26HW.git
!git clone -q https://github.com/uiuc-focal-lab/constraintflow.git
!cd constraintflow && git checkout -q 881e270 && git log -1 --oneline
!pip install -q "setuptools<81" onnx graphviz networkx "z3-solver==4.12.2.0" "antlr4-python3-runtime==4.9.2" typer
!pip install -q --no-deps -e ./constraintflow

881e270 (HEAD) Add no rewrite flag to cli for compilation
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 1.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 49.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for constraintflow (pyproject.toml) ... done


In [2]:
%cd CCS521FA26HW/hw2/q5
!ls

/content/CCS521FA26HW/hw2/q5
deeppoly.cf  mnist_relu_3_100.onnx  README.md  run_cf.py  train_mnist.py


In [3]:
%%writefile dualpoly.cf
def Shape as (Float l, Float u, PolyExp L, PolyExp U, PolyExp La){[(curr[l]<=curr),(curr[u]>=curr),(curr[L]<=curr),(curr[U]>=curr),(curr[La]<=curr)]};

func simplify_lower(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[l]) : (coeff * n[u]);
func simplify_upper(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[u]) : (coeff * n[l]);

func replace_lower(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[L]) : (coeff * n[U]);
func replace_upper(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[U]) : (coeff * n[L]);

func replace_lower_a(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[La]) : (coeff * n[U]);
func replace_upper_a(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[U]) : (coeff * n[La]);

func priority(Neuron n) = n[layer];
func priority2(Neuron n, Float c) = -n[layer];
func stop(Neuron n) = false;
func stop_traverse(Neuron n, Float c) = false;

func backsubs_lower(PolyExp e, Neuron n) = (e.traverse(backward, priority2, stop_traverse, replace_lower){e <= n}).map(simplify_lower);
func backsubs_upper(PolyExp e, Neuron n) = (e.traverse(backward, priority2, stop_traverse, replace_upper){e >= n}).map(simplify_upper);

func backsubs_lower_a(PolyExp e, Neuron n) = (e.traverse(backward, priority2, stop_traverse, replace_lower_a){e <= n}).map(simplify_lower);
func backsubs_upper_a(PolyExp e, Neuron n) = (e.traverse(backward, priority2, stop_traverse, replace_upper_a){e >= n}).map(simplify_upper);


func relu(Float x) = x >= 0.0 ? x : 0.0;
func relu_lower(Neuron n) = n[l] + n[u] >= 0.0 ? n : 0.0;
func relu_lower_a(Neuron n) = n[l] >= 0.0 ? n : (n[u] <= 0.0 ? 0.0 : (n[l] + n[u] >= 0.0 ? 0.0 : n));
func relu_upper(Neuron n) = n[l] >= 0.0 ? n : (n[u] <= 0.0 ? 0.0 : (((n[u]) / ((n[u]) - (n[l]))) * n) - (((n[u]) * (n[l])) / ((n[u]) - (n[l]))));

transformer dualpoly{
    Affine -> (max(backsubs_lower(prev.dot(curr[weight]) + curr[bias], curr), backsubs_lower_a(prev.dot(curr[weight]) + curr[bias], curr)),
               min(backsubs_upper(prev.dot(curr[weight]) + curr[bias], curr), backsubs_upper_a(prev.dot(curr[weight]) + curr[bias], curr)),
               prev.dot(curr[weight]) + curr[bias],
               prev.dot(curr[weight]) + curr[bias],
               prev.dot(curr[weight]) + curr[bias]);
    Relu -> (relu(prev[l]), relu(prev[u]), relu_lower(prev), relu_upper(prev), relu_lower_a(prev));
}

flow(forward, priority, stop, dualpoly);

Writing dualpoly.cf


In [4]:
%%writefile deeppoly_hardtanh.cf
def Shape as (Float l, Float u, PolyExp L, PolyExp U){[(curr[l]<=curr),(curr[u]>=curr),(curr[L]<=curr),(curr[U]>=curr)]};

func simplify_lower(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[l]) : (coeff * n[u]);
func simplify_upper(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[u]) : (coeff * n[l]);

func replace_lower(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[L]) : (coeff * n[U]);
func replace_upper(Neuron n, Float coeff) = (coeff >= 0.0) ? (coeff * n[U]) : (coeff * n[L]);

func priority(Neuron n) = n[layer];
func priority2(Neuron n, Float c) = -n[layer];
func stop(Neuron n) = false;
func stop_traverse(Neuron n, Float c) = false;

func backsubs_lower(PolyExp e, Neuron n) = (e.traverse(backward, priority2, stop_traverse, replace_lower){e <= n}).map(simplify_lower);
func backsubs_upper(PolyExp e, Neuron n) = (e.traverse(backward, priority2, stop_traverse, replace_upper){e >= n}).map(simplify_upper);

func hardtanh(Float x) = x >= 1.0 ? 1.0 : (x + 1.0 <= 0.0 ? (0.0 - 1.0) : x);

func ht_lower(Neuron n) = n[u] + 1.0 <= 0.0 ? (0.0 - 1.0) :
                          (n[l] >= 1.0 ? 1.0 :
                          (n[l] + 1.0 >= 0.0 ?
                              (n[u] <= 1.0 ? n :
                                  (((1.0 - n[l]) / (n[u] - n[l])) * n) + (n[l] - (((1.0 - n[l]) / (n[u] - n[l])) * n[l]))) :
                              (n[u] <= 1.0 ?
                                  ((n[u] + n[l] + 2.0 >= 0.0) ? n : (0.0 - 1.0)) :
                                  ((2.0 / (n[u] + 1.0)) * n) + (1.0 - ((2.0 / (n[u] + 1.0)) * n[u])))));

func ht_upper(Neuron n) = n[u] + 1.0 <= 0.0 ? (0.0 - 1.0) :
                          (n[l] >= 1.0 ? 1.0 :
                          (n[u] <= 1.0 ?
                              (n[l] + 1.0 >= 0.0 ? n :
                                  (((n[u] + 1.0) / (n[u] - n[l])) * n) + ((0.0 - 1.0) - (((n[u] + 1.0) / (n[u] - n[l])) * n[l]))) :
                              (n[l] + 1.0 >= 0.0 ?
                                  (((1.0 - n[l]) >= (n[u] - 1.0)) ? n : 1.0) :
                                  ((2.0 / (1.0 - n[l])) * n) + ((0.0 - 1.0) - ((2.0 / (1.0 - n[l])) * n[l])))));

transformer deeppoly{
    Affine -> (backsubs_lower(prev.dot(curr[weight]) + curr[bias], curr),
               backsubs_upper(prev.dot(curr[weight]) + curr[bias], curr),
               prev.dot(curr[weight]) + curr[bias],
               prev.dot(curr[weight]) + curr[bias]);
    HardTanh -> (hardtanh(prev[l]), hardtanh(prev[u]), ht_lower(prev), ht_upper(prev));
}

flow(forward, priority, stop, deeppoly);

Writing deeppoly_hardtanh.cf


In [5]:
!constraintflow provesound deeppoly.cf 2>&1 | grep -E 'Proved|unsound'

Proved Affine
Proved Relu


In [6]:
!constraintflow provesound dualpoly.cf 2>&1 | grep -E 'Proved|unsound'

Proved Affine
Proved Relu


In [7]:
!constraintflow provesound deeppoly_hardtanh.cf 2>&1 | grep -E 'Proved|unsound'

Proved Affine
Proved HardTanh


In [8]:
!CF_NO_PARALLEL=1 python run_cf.py deeppoly.cf --num-images 100 --eps 0.005 0.01 0.02 0.03 0.05 2>&1 | grep -v -i -E 'warn|pkg_resources'

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.40MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 128kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 19.0MB/s]
Network: mnist_relu_3_100.onnx
Clean accuracy on 100 images: 99.00%
     eps   verified   time/img (s)
   0.005     99.00%         0.4280
    0.01     94.00%         0.4301
    0.02     83.00%         0.4252
    0.03     49.00%         0.4317
    0.05      2.00%         0.4556


In [9]:
!CF_NO_PARALLEL=1 python run_cf.py dualpoly.cf --init La=L --num-images 100 --eps 0.005 0.01 0.02 0.03 0.05 2>&1 | grep -v -i -E 'warn|pkg_resources'

Network: mnist_relu_3_100.onnx
Clean accuracy on 100 images: 99.00%
     eps   verified   time/img (s)
   0.005     99.00%         0.8402
    0.01     94.00%         0.8256
    0.02     83.00%         0.8472
    0.03     49.00%         0.8522
    0.05      2.00%         0.8290


In [10]:
%%writefile compare_bounds.py
import contextlib, io, os, sys
import numpy as np
import torch
from torchvision import datasets, transforms

build_dir, out = sys.argv[1], sys.argv[2]
sys.path.insert(0, os.path.abspath(build_dir))
from main import run

test = datasets.MNIST("./data", train=False, download=True, transform=transforms.ToTensor())
X = torch.stack([test[i][0] for i in range(100)])
y = torch.tensor([test[i][1] for i in range(100)])
res = {}
for eps in [0.02, 0.03, 0.05]:
    lbs = []
    for i in range(100):
        with contextlib.redirect_stdout(io.StringIO()):
            lb, _ = run("mnist_relu_3_100.onnx", 1, eps, X[i:i + 1], y[i:i + 1], dataset="mnist",
                        train=False, print_intermediate_results=False, no_sparsity=False)
        lbs.append(lb.flatten().detach().cpu().numpy())
    res[str(eps)] = np.stack(lbs)
np.savez(out, **res)

Writing compare_bounds.py


In [11]:
!CF_NO_PARALLEL=1 python compare_bounds.py build_deeppoly lb_deeppoly.npz 2>&1 | grep -v -i -E 'warn|pkg_resources'
!CF_NO_PARALLEL=1 python compare_bounds.py build_dualpoly lb_dualpoly.npz 2>&1 | grep -v -i -E 'warn|pkg_resources'

In [12]:
import numpy as np
a, b = np.load("lb_deeppoly.npz"), np.load("lb_dualpoly.npz")
print(f"{'eps':>5} | {'DualPoly >= DeepPoly':>20} | {'bounds tighter':>15} | {'max gain':>8} | {'verified DP / Dual':>18}")
for k in a.files:
    d = b[k] - a[k]
    print(f"{k:>5} | {str(bool((d >= -1e-5).all())):>20} | {f'{(d > 1e-4).sum()} of {d.size}':>15} | "
          f"{d.max():8.4f} | {f'{(a[k] >= 0).all(1).sum()} / {(b[k] >= 0).all(1).sum()}':>18}")

  eps | DualPoly >= DeepPoly |  bounds tighter | max gain | verified DP / Dual
 0.02 |                 True |       33 of 900 |   0.0109 |            83 / 83
 0.03 |                 True |        0 of 900 |   0.0001 |            49 / 49
 0.05 |                 True |        1 of 900 |   1.0010 |              2 / 2
